# EDA: Suction-Line Restriction

## Data dictionary
Same 24 sensor columns + `Datetime` as every Simulated-dataset file — see notebook 01
for the full breakdown.

## Why this fault is a meaningful contrast to liquid-line restriction

Both are physical restrictions (a blockage/narrowing), not charge-level or fouling
faults — but at opposite ends of the refrigerant loop. Liquid-line restriction sits
between the condenser and the expansion device (high-pressure liquid line).
Suction-line restriction sits between the evaporator outlet and the compressor inlet
(low-pressure vapor line) — restricting the path refrigerant takes on its way *back*
to the compressor, after already absorbing heat at the evaporator.

## Hypothesis (before looking at any data)

- A restriction on the suction line should cause a pressure drop *at* the compressor
  inlet specifically — expect `RTU_REFG_SUCT_PRES` to show a real, direct effect (this
  is literally the line being restricted), likely more directly than liquid-line
  restriction's more indirect effect on the same column.
- Real, falsifiable prediction carried over from notebook 05's closing question: does
  this fault show a smooth, scaling trend (like the charge-level faults) or a
  threshold effect (like liquid-line restriction)? Genuinely unknown — restricting the
  low-pressure vapor line is a different physical mechanism than restricting the
  high-pressure liquid line (compressible vapor vs. liquid refrigerant behave very
  differently under a restriction), so there's no strong prior reason to expect the
  same shape either way.
- Severity here is in bar again, but a different spread (1/3/6/9, vs. liquid-line's
  1/4/8/10) — worth checking whether that reflects different severity-selection choices
  by LBNL for this fault specifically, not something to assume matches the other
  restriction fault's scale.
- Given liquid-line restriction's stage-2 filtering revealed a real, large staging
  confound in capacity, apply that filter here from the start (informed by the last
  five notebooks) rather than treating it as a late-stage "check" step — though still
  worth reporting both filtered and unfiltered numbers for the same honesty/comparison
  reasons as before.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from src.features.effect_size import cohens_d  # noqa: E402
from src.features.filtering import stage2_only  # noqa: E402

files = {
    "baseline": "../data/raw/RTU_sim_baseline.csv",
    "suctionpipe01bar": "../data/raw/RTU_sim_suctionpipe01bar.csv",
    "suctionpipe03bar": "../data/raw/RTU_sim_suctionpipe03bar.csv",
    "suctionpipe06bar": "../data/raw/RTU_sim_suctionpipe06bar.csv",
    "suctionpipe09bar": "../data/raw/RTU_sim_suctionpipe09bar.csv",
}

dfs = {label: pd.read_csv(fname) for label, fname in files.items()}

for _label, df in dfs.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

for label, df in dfs.items():
    print(f"{label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

baseline: shape=(143941, 25), missing_values=0
suctionpipe01bar: shape=(143941, 25), missing_values=0
suctionpipe03bar: shape=(143941, 25), missing_values=0
suctionpipe06bar: shape=(143941, 25), missing_values=0
suctionpipe09bar: shape=(143941, 25), missing_values=0


## Load confirmed

All 5 files: shape=(143941, 25), 0 missing values.

In [2]:
severity_order = ["baseline", "suctionpipe01bar", "suctionpipe03bar", "suctionpipe06bar", "suctionpipe09bar"]

restriction_cols = ["RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_REFG_DISC_PRES", "RTU_TOT_CAPA"]

stage2_dfs = {label: stage2_only(df) for label, df in dfs.items()}

summary_unfiltered = pd.DataFrame({
    label: df[restriction_cols].mean()
    for label, df in dfs.items()
}).T.loc[severity_order]

summary_filtered = pd.DataFrame({
    label: df[restriction_cols].mean()
    for label, df in stage2_dfs.items()
}).T.loc[severity_order]

pct_change_unfiltered = (summary_unfiltered / summary_unfiltered.loc["baseline"] - 1) * 100
pct_change_filtered = (summary_filtered / summary_filtered.loc["baseline"] - 1) * 100

print("Unfiltered % change:")
print(pct_change_unfiltered)
print("\nFiltered (stage-2 only) % change:")
print(pct_change_filtered)

Unfiltered % change:
                  RTU_REFG_SUCT_PRES  RTU_REFG_SUCT_TEMP  RTU_REFG_DISC_PRES  \
baseline                    0.000000            0.000000            0.000000   
suctionpipe01bar           -2.118903           -2.283327           -0.211218   
suctionpipe03bar          -18.042222          -20.117488           -1.658996   
suctionpipe06bar          -29.874368          -35.131391           -2.625653   
suctionpipe09bar          -56.417195          -77.718516           -6.214599   

                  RTU_TOT_CAPA  
baseline              0.000000  
suctionpipe01bar     -0.961311  
suctionpipe03bar     -6.568925  
suctionpipe06bar    -12.024396  
suctionpipe09bar    -31.800810  

Filtered (stage-2 only) % change:
                  RTU_REFG_SUCT_PRES  RTU_REFG_SUCT_TEMP  RTU_REFG_DISC_PRES  \
baseline                    0.000000            0.000000            0.000000   
suctionpipe01bar           -2.874762           -3.237821           -0.980044   
suctionpipe03bar         

## Finding: suction-line restriction is the strongest, cleanest, most severe fault
## found across the entire Simulated dataset

Unlike liquid-line restriction's threshold shape, every column here is **cleanly
monotonic** across all four severities (1/3/6/9 bar), both filtered and unfiltered:

| Severity | SUCT_PRES unfilt | SUCT_PRES filt | SUCT_TEMP unfilt | SUCT_TEMP filt | CAPA unfilt | CAPA filt |
|---|---|---|---|---|---|---|
| 1 bar | -2.12% | -2.87% | -2.28% | -3.24% | -0.96% | -2.55% |
| 3 bar | -18.04% | -20.43% | -20.12% | -24.18% | -6.57% | -18.23% |
| 6 bar | -29.87% | -31.91% | -35.13% | -40.15% | -12.02% | -29.87% |
| 9 bar | -56.42% | -52.53% | -77.72% | -75.70% | -31.80% | -50.62% |

**Directly confirms the hypothesis**: `RTU_REFG_SUCT_PRES` shows the largest, most
direct effect of any restriction/fouling fault examined — mechanistically sensible,
since this line is literally what's being restricted, feeding straight into the
compressor's suction port.

**Capacity's effect is the largest found in the entire dataset**: -50.62% at 9 bar
(filtered) — more than 2.7x liquid-line restriction's peak (-7.49%) and nearly 3x
evaporator fouling's peak (-18.49%), the previous strongest capacity effect.

**Filtering behaves oppositely here compared to liquid-line restriction**: there,
filtering *shrank* the apparent effect (revealing inflated staging noise). Here,
filtering *grows* the effect at every severity except 9 bar's suction pressure
(-56.42% unfiltered vs -52.53% filtered — the one column/severity where unfiltered was
slightly larger). This suggests stage-1/stage-2 blending was diluting this fault's
visibility in the raw mean, not manufacturing a false positive — the opposite
direction of confound from what liquid-line restriction showed.

**Practical implication for modeling**: suction-line restriction looks like it will be
the easiest of the six Simulated faults to detect and classify — every candidate
signal moves cleanly, strongly, and monotonically with severity, with no threshold
gaps or non-monotonic reversals to complicate classification.

In [3]:
d_1_vs_3 = cohens_d(
    stage2_dfs["suctionpipe01bar"]["RTU_TOT_CAPA"],
    stage2_dfs["suctionpipe03bar"]["RTU_TOT_CAPA"],
)
print(f"Cohen's d, 1bar vs 3bar (RTU_TOT_CAPA, stage-2 filtered): {d_1_vs_3:.3f}")

Cohen's d, 1bar vs 3bar (RTU_TOT_CAPA, stage-2 filtered): 4.439


## Cohen's d confirms: even the smallest severity gap here is enormous

`suctionpipe01bar` vs `suctionpipe03bar` (stage-2 `RTU_TOT_CAPA`): **d = 4.439** — far
beyond the conventional "large" threshold (0.8), and the largest effect size found
across any fault or severity comparison in this entire EDA. Even the mildest severity
step tested for this fault (1 bar → 3 bar) is more dramatically separated than the most
extreme comparisons in every other fault examined (undercharge d≈1.33-1.44, liquid-line
restriction's largest single jump d=1.445).

**Conclusion**: suction-line restriction is not just cleanly monotonic — it produces an
overwhelmingly strong signal at every severity level tested, with no weak spots. This
is the opposite extreme from condenser fouling's near-zero adjacent-severity separation
(d=-0.037 at 30% vs 40%) found earlier in this project.

## Summary: suction-line restriction EDA

**Hypothesis confirmed strongly**: as the line directly feeding the compressor's
suction port, restricting it produces the largest, most direct, most cleanly monotonic
effect found across every fault in the Simulated dataset — no threshold shape (unlike
liquid-line restriction), no non-monotonic wobble (unlike overcharge or condenser
fouling), no weak adjacent-severity gaps (unlike condenser fouling's capacity).

**Every signal checked is monotonic**, filtered and unfiltered, across all four
severities (1/3/6/9 bar). `RTU_TOT_CAPA`'s effect at 9 bar (-50.62% filtered) is the
largest capacity impact found in this entire EDA — nearly 3x evaporator fouling's
previous record.

**Filtering behaves oppositely from liquid-line restriction**: there, filtering
revealed inflated staging noise (effect shrank). Here, filtering reveals the true
effect was being diluted, not inflated (effect grew). A reminder that stage-2
filtering's direction of correction isn't predictable per fault — must be checked
each time, not assumed from a prior fault's result.

**Cohen's d = 4.439 at the smallest severity gap tested (1 vs 3 bar)** — the largest
effect size found across all six Simulated-dataset faults, confirming this isn't just
"looks clean on a table" but genuinely, overwhelmingly separated at the distribution
level.

**Practical implication for modeling**: of the six Simulated-dataset faults, this one
should be the easiest to classify at any severity, including the mildest tested. No
open questions or unresolved wobbles remain for this fault, unlike overcharge
(non-monotonic discharge pressure), condenser fouling (weak capacity separation), or
liquid-line restriction (threshold effect, non-monotonic discharge pressure).